In [68]:
import pandas as pd
from pandas.tseries.offsets import BDay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import joblib
import numpy as np

In [69]:
df = pd.read_csv("data/job_time_data.csv")

In [70]:
df['printing_date'] = pd.to_datetime(df['printing_date'], format='%d/%m/%Y')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], format='%d/%m/%Y')
df['created_date'] = df['printing_date'] - BDay(4)
df['delivery_gap_days'] = (df['delivery_date'] - df['created_date']).dt.days
df['special_request'] = df['delivery_gap_days'].apply(lambda x: 'Yes' if x > 25 else 'No')
df['created_month'] = df['created_date'].dt.month
df['created_weekday'] = df['created_date'].dt.weekday

In [71]:
X = df[['quantity', 'no_of_impressions', 'color', 'finishing', 'created_month', 'created_weekday']]
y = df['delivery_gap_days']

categorical_features = ['finishing']
numeric_features = ['quantity', 'no_of_impressions', 'color', 'created_month', 'created_weekday']

In [72]:
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [74]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [75]:
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['auto', 'sqrt', 'log2']
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=2
)

In [76]:
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

Fitting 3 folds for each of 324 candidates, totalling 972 fits
Best parameters: {'regressor__max_depth': 10, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}


d:\Projects\RN_Printing-ERP\backend\prediction_model\.venv\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
324 fits failed out of a total of 972.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
213 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Projects\RN_Printing-ERP\backend\prediction_model\.venv\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Projects\RN_Printing-ERP\backend\prediction_model\.venv\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "d:\Projects\RN_Printing-ERP\backend\prediction_model\.ve

In [77]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 3.107496227777771
RMSE: 8.46852260630847
R2 Score: -0.07520052673999422


In [78]:
joblib.dump(best_model, "random_forest_jobtime_model_tuned.pkl")
print("Tuned model saved as random_forest_jobtime_model_tuned.pkl")

Tuned model saved as random_forest_jobtime_model_tuned.pkl
